### Problem Statement : Hospital Patient Data Analysis

In [2]:
import pandas as pd
Patient_df = pd.read_csv('Patient_Data.csv')
Billing_df= pd.read_csv('Billing_Data.csv')
# Data Summary
print(Patient_df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes
None


In [5]:
# Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].
billing_cols= Patient_df[['PatientID', 'Department', 'Doctor', 'BillAmount']]
print(billing_cols.head())

   PatientID   Department     Doctor  BillAmount
0        101   Cardiology  Dr. Smith      5000.0
1        102    Neurology   Dr. John         NaN
2        103  Orthopedics    Dr. Lee      7500.0
3        104   Cardiology  Dr. Smith      6200.0
4        105  Dermatology   Dr. Rose         NaN


In [6]:
# Drop administrative columns like ['ReceptionistID', 'CheckInTime'].
Patient_df= Patient_df.drop(['ReceptionistID', 'CheckInTime'],axis=1)
print(Patient_df)

   PatientID     Name   Department     Doctor  BillAmount
0        101    Alice   Cardiology  Dr. Smith      5000.0
1        102      Bob    Neurology   Dr. John         NaN
2        103  Charlie  Orthopedics    Dr. Lee      7500.0
3        104    David   Cardiology  Dr. Smith      6200.0
4        105      Eva  Dermatology   Dr. Rose         NaN
5        101    Alice   Cardiology  Dr. Smith      5000.0


In [7]:
# Use groupby to find total bill amount per department.
dept_bill = Patient_df.groupby('Department')['BillAmount'].sum()
print(dept_bill)

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


In [9]:
# Remove duplicate patient records based on PatientID
Patient_df = Patient_df.drop_duplicates(subset='PatientID')
print(Patient_df)

   PatientID     Name   Department     Doctor  BillAmount
0        101    Alice   Cardiology  Dr. Smith      5000.0
1        102      Bob    Neurology   Dr. John         NaN
2        103  Charlie  Orthopedics    Dr. Lee      7500.0
3        104    David   Cardiology  Dr. Smith      6200.0
4        105      Eva  Dermatology   Dr. Rose         NaN


In [14]:
# Fill missing BillAmount values with the mean bill amount

Patient_df['BillAmount']=pd.to_numeric(Patient_df['BillAmount'])
Patient_df['BillAmount']=Patient_df['BillAmount'].fillna(Patient_df['BillAmount'].mean())
print(Patient_df['BillAmount'])

0    5000.000000
1    6233.333333
2    7500.000000
3    6200.000000
4    6233.333333
Name: BillAmount, dtype: float64


C:\Users\Mitali Patil\AppData\Local\Temp\ipykernel_4104\2184193410.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Patient_df['BillAmount']=pd.to_numeric(Patient_df['BillAmount'])
C:\Users\Mitali Patil\AppData\Local\Temp\ipykernel_4104\2184193410.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Patient_df['BillAmount']=Patient_df['BillAmount'].fillna(Patient_df['BillAmount'].mean())


In [15]:
# Merge the billing dataset with patient dataset on PatientID.
merged_df = pd.merge(Patient_df,Billing_df, on='PatientID', how='inner')
print(merged_df)

   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000              2000   
1        102      Bob    Neurology   Dr. John  6233.333333              1500   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              2500   
3        104    David   Cardiology  Dr. Smith  6200.000000              3000   
4        105      Eva  Dermatology   Dr. Rose  6233.333333              1000   

   FinalAmount  
0         3000  
1         3500  
2         5000  
3         3200  
4         4000  


In [17]:
# Concatenate an additional DataFrame that contains new patients for the current week (row-wise).
new_patients = pd.DataFrame({
    'PatientID':[501,502],
    'Department':['Cardiology','Neurology'],
    'Doctor':['Dr. Mehta','Dr. Shah'],
    'BillAmount':[8000,12000]
})

Patient_df = pd.concat([Patient_df,new_patients], axis=0)

print(Patient_df)

   PatientID     Name   Department     Doctor    BillAmount
0        101    Alice   Cardiology  Dr. Smith   5000.000000
1        102      Bob    Neurology   Dr. John   6233.333333
2        103  Charlie  Orthopedics    Dr. Lee   7500.000000
3        104    David   Cardiology  Dr. Smith   6200.000000
4        105      Eva  Dermatology   Dr. Rose   6233.333333
0        501      NaN   Cardiology  Dr. Mehta   8000.000000
1        502      NaN    Neurology   Dr. Shah  12000.000000


In [18]:
# Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

billing_extra = pd.DataFrame({
    'InsuranceCovered':[2000,1500,1000],
    'FinalAmount':[6000,8500,9000]
})

final_df = pd.concat([merged_df,billing_extra], axis=1)

print(final_df.head())

   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000              2000   
1        102      Bob    Neurology   Dr. John  6233.333333              1500   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              2500   
3        104    David   Cardiology  Dr. Smith  6200.000000              3000   
4        105      Eva  Dermatology   Dr. Rose  6233.333333              1000   

   FinalAmount  InsuranceCovered  FinalAmount  
0         3000            2000.0       6000.0  
1         3500            1500.0       8500.0  
2         5000            1000.0       9000.0  
3         3200               NaN          NaN  
4         4000               NaN          NaN  
